In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/loan-prediction-dataset-2025/loan_dataset_20000.csv
/kaggle/input/playground-series-s5e11/sample_submission.csv
/kaggle/input/playground-series-s5e11/train.csv
/kaggle/input/playground-series-s5e11/test.csv


In [2]:
N_FOLDS = 5
SEED = 42

In [3]:
INPUT_DIR = '/kaggle/input/playground-series-s5e11'
train = pd.read_csv(f'{INPUT_DIR}/train.csv')
train_org = pd.read_csv('/kaggle/input/loan-prediction-dataset-2025/loan_dataset_20000.csv')
test = pd.read_csv(f'{INPUT_DIR}/test.csv')
TARGET = train.columns[-1]
test[TARGET] = -1
# combine

In [4]:
FEATURES = list(train.columns[1:-1])
print(f'FEATURES_{len(FEATURES)}: {FEATURES}, TARGET: {TARGET}')

FEATURES_11: ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade'], TARGET: loan_paid_back


In [5]:
train_org = train_org[FEATURES + [TARGET]]
train_org[TARGET] = train_org[TARGET].astype('float64')

# train_org = train_org[FEATURES + [TARGET]].reset_index(drop=True)
# train     = train[FEATURES + [TARGET]].reset_index(drop=True)
# test      = test[FEATURES].reset_index(drop=True)         

# n_org = len(train_org)
# n_tr  = len(train)
# n_te  = len(test)

combine = pd.concat([train_org, train.drop(columns='id'), test.drop(columns='id')], axis=0)

In [6]:
CATS = []
NUMS = []

for c in FEATURES:
    t='CAT'
    if train[c].dtype=='object':
        CATS.append(c)
    else:
        NUMS.append(c)
        t='NUM'

    n = train[c].nunique()
    na = train[c].isna().sum()
    print(f'[{t}] {c} has {n} unique and {na} NA')

print('CATS:', CATS)
print('NUMS:', NUMS)

[NUM] annual_income has 119728 unique and 0 NA
[NUM] debt_to_income_ratio has 526 unique and 0 NA
[NUM] credit_score has 399 unique and 0 NA
[NUM] loan_amount has 111570 unique and 0 NA
[NUM] interest_rate has 1454 unique and 0 NA
[CAT] gender has 3 unique and 0 NA
[CAT] marital_status has 4 unique and 0 NA
[CAT] education_level has 5 unique and 0 NA
[CAT] employment_status has 5 unique and 0 NA
[CAT] loan_purpose has 8 unique and 0 NA
[CAT] grade_subgrade has 30 unique and 0 NA
CATS: ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
NUMS: ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']


In [7]:
CATS1 = []
SIZES = {}

for c in CATS:
    n=f'{c}2'
    if c in NUMS+CATS:
        n=f'{c}2'
        CATS1.append(n)
    combine[n],_ = combine[c].factorize()
    SIZES[n] = combine[n].max()+1

    # combine[c] = combine[c].astype('int32')
    # combine[n] = combine[n].astype('int32')

print('NEW CATS:', CATS1)
print('CARDINALITY OF ALL CATS:', SIZES)

NEW CATS: ['gender2', 'marital_status2', 'education_level2', 'employment_status2', 'loan_purpose2', 'grade_subgrade2']
CARDINALITY OF ALL CATS: {'gender2': 3, 'marital_status2': 4, 'education_level2': 5, 'employment_status2': 5, 'loan_purpose2': 8, 'grade_subgrade2': 30}


In [8]:
from itertools import combinations

INTER = []

for col1, col2 in combinations(FEATURES, 2):
    new_col_name = f'{col1}_{col2}'
    INTER.append(new_col_name)
    for df in [combine]:
        df[new_col_name] = df[col1].astype(str) + '_' + df[col2].astype(str)
        
print(f'{len(INTER)} Features.')

55 Features.


In [9]:
# BINS = []
# for col1, col2 in combinations(NUMS, 2):
#     n=f'{col1}_{col2}'
#     for df in [combine]:
#         df[f'{n}_int'] = df[col1] * df[col2]
#         df[f'{n}_add'] = df[col1] + df[col2]
#         df[f'{n}_sub'] = df[col1] - df[col2]
#         df[f'{n}_div'] = df[col1] / (df[col2] + 1e-8)   # add eps to avoid /0
#         df[f'{n}_logdiv'] = np.log1p(df[col1]) - np.log1p(df[col2])
#         df[f'{n}_L2']   = np.sqrt(df[col1]**2 + df[col2]**2)
#         df[f'{n}_L1']   = np.abs(df[col1]) + np.abs(df[col2])
#         df[f'{n}_dist'] = np.abs(df[col1] - df[col2])
#         df[f'{n}_min'] = np.minimum(df[col1], df[col2])
#         df[f'{n}_max'] = np.maximum(df[col1], df[col2])
#         df[f'{n}_avg'] = (df[col1] + df[col2]) / 2
#         df[f'{n}_sq1'] = df[col1]**2
#         df[f'{n}_sq2'] = df[col2]**2
#         df[f'{n}_poly'] = 2 * df[col1] * df[col2]   # 2xy term
#         df[f'{n}_cos'] = np.cos(df[col1] - df[col2])
#         df[f'{n}_sin'] = np.sin(df[col1] - df[col2])
#         df[f'{n}_gt']  = (df[col1] > df[col2]).astype(int)
#         df[f'{n}_eq']  = (np.isclose(df[col1], df[col2])).astype(int)
#         # df[f'{col1}_freq_X_{col2}'] = df[f'{col1}_freq'] * df[col2]



In [10]:
train_org_n = combine.iloc[:len(train_org)]
train_n = combine.iloc[len(train_org):len(train)+len(train_org)]
test_n = combine.iloc[len(train)+len(train_org):]

In [11]:
TE = []
for c in FEATURES:
    tmp = train_org_n.groupby(c)[TARGET].mean()
    tmp_sum = train_org_n.groupby(c)[TARGET].sum()
    tmp_cnt = train_org_n.groupby(c).size()
    
    n = f'TE_{c}'
    n_s = f'TE_sum_{c}'
    n_c = f'TE_count_{c}'
    print(f'{n} , {n_c}', end='')
    tmp.name = n
    tmp_cnt.name = n_c
    tmp_sum.name = n_s

    stats = (pd.concat([tmp,  
                       # tmp_sum,
                        tmp_cnt
                      ]
                      , axis=1,).reset_index().rename(columns={'index': c})) 
    train_org_n = train_org_n.merge(stats, on=c, how='left')
    
    train_n = train_n.merge(stats, on=c, how='left')
    
    test_n = test_n.merge(stats, on=c, how='left')
    
    TE.append(n)
    TE.append(n_c)

TE_annual_income , TE_count_annual_incomeTE_debt_to_income_ratio , TE_count_debt_to_income_ratioTE_credit_score , TE_count_credit_scoreTE_loan_amount , TE_count_loan_amountTE_interest_rate , TE_count_interest_rateTE_gender , TE_count_genderTE_marital_status , TE_count_marital_statusTE_education_level , TE_count_education_levelTE_employment_status , TE_count_employment_statusTE_loan_purpose , TE_count_loan_purposeTE_grade_subgrade , TE_count_grade_subgrade

In [12]:
BINS = []  # final list of binned column names
q_list = [5, 10, 15, 20, 25]

for col in NUMS:  # original numeric columns only
    for q in q_list:
        col_name = f"{col}_bin{q}"
        
        # 1. learn bins on TRAIN_ORG (pure training data)
        try:
            _, bins = pd.qcut(
                train_org_n[col], q=q, retbins=True, labels=False, duplicates="drop"
            )
        except ValueError:  # all values identical
            bins = np.linspace(train_org_n[col].min(), train_org_n[col].max(), q + 1)

        # 2. apply identical bins to all three splits
        train_org_n[col_name] = pd.cut(
            train_org_n[col], bins=bins, labels=False, include_lowest=True
        ).astype(np.int8)

        train_n[col_name] = pd.cut(
            train_n[col], bins=bins, labels=False, include_lowest=True
        ).astype(np.int8)

        test_n[col_name] = pd.cut(
            test_n[col], bins=bins, labels=False, include_lowest=True
        ).astype(np.int8)

        BINS.append(col_name)

print(f"{len(BINS)} k-bins discretisation features created.")

25 k-bins discretisation features created.


In [13]:
from sklearn.base import BaseEstimator, TransformerMixin

class TargetEncoder(BaseEstimator, TransformerMixin):
    """
    Target Encoder that supports multiple aggregation functions,
    internal cross-validation for leakage prevention, and smoothing.

    Parameters
    ----------
    cols_to_encode : list of str
        List of column names to be target encoded.

    aggs : list of str, default=['mean']
        List of aggregation functions to apply. Any function accepted by
        pandas' `.agg()` method is supported, such as:
        'mean', 'std', 'var', 'min', 'max', 'skew', 'nunique', 
        'count', 'sum', 'median'.
        Smoothing is applied only to the 'mean' aggregation.

    cv : int, default=5
        Number of folds for cross-validation in fit_transform.

    smooth : float or 'auto', default='auto'
        The smoothing parameter `m`. A larger value puts more weight on the 
        global mean. If 'auto', an empirical Bayes estimate is used.
        
    drop_original : bool, default=False
        If True, the original columns to be encoded are dropped.
    """
    def __init__(self, cols_to_encode, aggs=['mean'], cv=5, smooth='auto', drop_original=False):
        self.cols_to_encode = cols_to_encode
        self.aggs = aggs
        self.cv = cv
        self.smooth = smooth
        self.drop_original = drop_original
        self.mappings_ = {}
        self.global_stats_ = {}

    def fit(self, X, y):
        """
        Learn mappings from the entire dataset.
        These mappings are used for the transform method on validation/test data.
        """
        temp_df = X.copy()
        temp_df['target'] = y

        # Learn global statistics for each aggregation
        for agg_func in self.aggs:
            self.global_stats_[agg_func] = y.agg(agg_func)

        # Learn category-specific mappings
        for col in self.cols_to_encode:
            self.mappings_[col] = {}
            for agg_func in self.aggs:
                mapping = temp_df.groupby(col)['target'].agg(agg_func)
                self.mappings_[col][agg_func] = mapping
        
        return self

    def transform(self, X):
        """
        Apply learned mappings to the data.
        Unseen categories are filled with global statistics.
        """
        X_transformed = X.copy()
        for col in self.cols_to_encode:
            for agg_func in self.aggs:
                new_col_name = f'TE_{col}_{agg_func}'
                map_series = self.mappings_[col][agg_func]
                X_transformed[new_col_name] = X[col].map(map_series)
                X_transformed[new_col_name].fillna(self.global_stats_[agg_func], inplace=True)
        
        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)
            
        return X_transformed

    def fit_transform(self, X, y):
        """
        Fit and transform the data using internal cross-validation to prevent leakage.
        """
        # First, fit on the entire dataset to get global mappings for transform method
        self.fit(X, y)

        # Initialize an empty DataFrame to store encoded features
        encoded_features = pd.DataFrame(index=X.index)
        
        kf = StratifiedKFold(n_splits=self.cv, shuffle=True, random_state=42)

        for train_idx, val_idx in kf.split(X, y):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_val = X.iloc[val_idx]
            
            temp_df_train = X_train.copy()
            temp_df_train['target'] = y_train

            for col in self.cols_to_encode:
                # --- Calculate mappings only on the training part of the fold ---
                for agg_func in self.aggs:
                    new_col_name = f'TE_{col}_{agg_func}'
                    
                    # Calculate global stat for this fold
                    fold_global_stat = y_train.agg(agg_func)
                    
                    # Calculate category stats for this fold
                    mapping = temp_df_train.groupby(col)['target'].agg(agg_func)

                    # --- Apply smoothing only for 'mean' aggregation ---
                    if agg_func == 'mean':
                        counts = temp_df_train.groupby(col)['target'].count()
                        
                        m = self.smooth
                        if self.smooth == 'auto':
                            # Empirical Bayes smoothing
                            variance_between = mapping.var()
                            avg_variance_within = temp_df_train.groupby(col)['target'].var().mean()
                            if variance_between > 0:
                                m = avg_variance_within / variance_between
                            else:
                                m = 0  # No smoothing if no variance between groups
                        
                        # Apply smoothing formula
                        smoothed_mapping = (counts * mapping + m * fold_global_stat) / (counts + m)
                        encoded_values = X_val[col].map(smoothed_mapping)
                    else:
                        encoded_values = X_val[col].map(mapping)
                    
                    # Store encoded values for the validation fold
                    encoded_features.loc[X_val.index, new_col_name] = encoded_values.fillna(fold_global_stat)

        # Merge with original DataFrame
        X_transformed = X.copy()
        for col in encoded_features.columns:
            X_transformed[col] = encoded_features[col]
            
        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)
            
        return X_transformed

In [14]:
# CATS = train.select_dtypes(include='object').columns.to_list()
# train[CATS] = train[CATS].astype('category')
# test[CATS] = test[CATS].astype('category')

# for c in CATS:
#     for df in [train, test]:
#         df[c], _ = df[c].factorize()

In [15]:
train_n.shape

(593994, 120)

In [16]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 5,
    'colsample_bytree': 0.5,
    'subsample': 0.8,
    'n_estimators': 10000,
    'learning_rate': 0.01,
    'early_stopping_rounds': 100,
    'random_state': 42,
    'n_jobs': -1,
    'device': 'cuda',
    'enable_categorical': True,
}
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
X = train_n.drop(columns=[TARGET])
y = train_n[TARGET]
print(f'X shape: {X.shape}')

# oof_preds = np.zeros(len(X))
# test_preds = np.zeros(len(test))


# for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
#     print(f'--- Fold {fold}/{N_FOLDS} ---')
    
#     X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
#     # X_test = test[FEATURES].copy()
#     X_test = test_n.drop(columns=TARGET).copy()

#     TE = TargetEncoder(cols_to_encode=INTER, cv=5, smooth='auto', aggs=['mean'], drop_original=True)
#     X_train = TE.fit_transform(X_train, y_train)
#     X_val = TE.transform(X_val)
#     X_test = TE.transform(X_test)

#     X_train[CATS] = X_train[CATS].astype('category')
#     X_val[CATS] = X_val[CATS].astype('category')
#     X_test[CATS] = X_test[CATS].astype('category')

#     model = XGBClassifier(**params)
    
#     model.fit(X_train, y_train,
#               eval_set=[(X_val, y_val)],
#               verbose=1000)

#     val_preds = model.predict_proba(X_val)[:, 1]
#     oof_preds[val_idx] = val_preds
    
#     fold_score = roc_auc_score(y_val, val_preds)
#     print(f'Fold {fold} AUC: {fold_score:.4f}')
#     test_preds += model.predict_proba(X_test)[:, 1] / N_FOLDS

# overall_auc = roc_auc_score(y, oof_preds)
# print(f'====================')
# print(f'Overall OOF AUC: {overall_auc:.4f}')
# print(f'====================')

X shape: (593994, 119)


In [17]:
# pd.DataFrame({'id': train.id, TARGET: oof_preds}).to_csv(f'oof_xgb_cv_{overall_auc}.csv', index=False)
# pd.DataFrame({'id': test.id, TARGET: test_preds}).to_csv(f'test_xgb_cv_{overall_auc}.csv', index=False)

In [18]:
# =====  LGBoost cell – TE fit inside, no external reuse  =====
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

lgb_params = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'num_leaves': 31, 'max_depth': 5, 'learning_rate': 0.01,
    'n_estimators': 10000, 'subsample': 0.8, 'colsample_bytree': 0.5,
    'reg_alpha': 0.001, 'reg_lambda': 0.001, 'random_state': 42,
    'n_jobs': -1, 'verbose': -1, 'early_stopping_rounds': 100
}

oof_lgb = np.zeros(len(X))
test_lgb = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f'--- LGB Fold {fold}/{N_FOLDS} ---')
    
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    X_te = test_n.drop(columns=[TARGET])

    # TargetEncoder fitted **inside** this cell only
    te = TargetEncoder(cols_to_encode=INTER, cv=5, smooth='auto',
                       aggs=['mean'], drop_original=True)
    X_tr = te.fit_transform(X_tr, y_tr)
    X_val = te.transform(X_val)
    X_te  = te.transform(X_te)

    # categorical dtype
    for col in CATS:
        X_tr[col] = X_tr[col].astype('category')
        X_val[col] = X_val[col].astype('category')
        X_te[col] = X_te[col].astype('category')

    train_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=CATS)
    val_data   = lgb.Dataset(X_val, label=y_val, reference=train_data)

    model = lgb.train(lgb_params, train_data, valid_sets=[val_data],
                      callbacks=[lgb.early_stopping(100), lgb.log_evaluation(1000)])

    val_pred = model.predict(X_val, num_iteration=model.best_iteration)
    oof_lgb[val_idx] = val_pred
    test_lgb += model.predict(X_te, num_iteration=model.best_iteration) / N_FOLDS

    print(f'LGB Fold {fold} AUC: {roc_auc_score(y_val, val_pred):.4f}')

overall_lgb = roc_auc_score(y, oof_lgb)
print(f'====================\nLGB OOF AUC: {overall_lgb:.4f}\n====================')

# save
pd.DataFrame({'id': train.id, TARGET: oof_lgb}) \
    .to_csv(f'oof_lgb_cv_{overall_lgb:.4f}.csv', index=False)
pd.DataFrame({'id': test.id, TARGET: test_lgb}) \
    .to_csv(f'test_lgb_cv_{overall_lgb:.4f}.csv', index=False)

--- LGB Fold 1/5 ---
Training until validation scores don't improve for 100 rounds
[1000]	valid_0's auc: 0.925818
Early stopping, best iteration is:
[1010]	valid_0's auc: 0.925827
LGB Fold 1 AUC: 0.9258
--- LGB Fold 2/5 ---
Training until validation scores don't improve for 100 rounds
[1000]	valid_0's auc: 0.92671
Early stopping, best iteration is:
[1009]	valid_0's auc: 0.926714
LGB Fold 2 AUC: 0.9267
--- LGB Fold 3/5 ---
Training until validation scores don't improve for 100 rounds
[1000]	valid_0's auc: 0.924287
Early stopping, best iteration is:
[961]	valid_0's auc: 0.924326
LGB Fold 3 AUC: 0.9243
--- LGB Fold 4/5 ---
Training until validation scores don't improve for 100 rounds
[1000]	valid_0's auc: 0.925409
Early stopping, best iteration is:
[1026]	valid_0's auc: 0.925418
LGB Fold 4 AUC: 0.9254
--- LGB Fold 5/5 ---
Training until validation scores don't improve for 100 rounds
[1000]	valid_0's auc: 0.925368
Early stopping, best iteration is:
[1037]	valid_0's auc: 0.925395
LGB Fold 5